# Exploración de datos raw y conformación de Procesamiento de Datos

Objetivos:
1. Explorar la estructura de los archivos en `data/raw` (alarmReport, dls_report, Historial).
2. Generar de forma automática los archivos **Procesamiento de datos** por año-mes en formato CSV.
3. Guardar los CSV en `data/processed` con el patrón `Procesamiento_de_Datos_YYYY-MM.csv`.

## 1. Exploración de la carpeta `data/raw`

In [1]:
import sys
from pathlib import Path

# Añadir raíz del proyecto al path para importar config (local vs cloud desde .env)
_nb_dir = Path(".").resolve()
_project_root = _nb_dir.parent if _nb_dir.name == "notebooks" else _nb_dir
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from config import RAW_DIR, PROCESSED_DIR, OVERWRITE_PROCESSED_FILES, RUN_ENV
import pandas as pd

# Constantes: nombres de columnas del dominio (fuente dls_report)
COL_READING = "Leyendo"
DLS_SHEET_NAME = "Resumen"
DLS_HEADER_ROW = 24

# Listar y clasificar archivos en data/raw
raw_file_paths = [f for f in RAW_DIR.glob("*") if f.is_file()]
alarm_report_names = [f.name for f in raw_file_paths if "alarmReport" in f.name]
dls_report_names = [f.name for f in raw_file_paths if "dls_report" in f.name]
historial_file_names = [f.name for f in raw_file_paths if "Historial" in f.name]
known_names = alarm_report_names + dls_report_names + historial_file_names
other_file_names = [f.name for f in raw_file_paths if f.name not in known_names]

print("Archivos en data/raw:")
print(f"  - alarmReport_*.xlsx: {len(alarm_report_names)} archivos")
print(f"  - dls_report_*.xlsx:  {len(dls_report_names)} archivos (fuente de Procesamiento de Datos)")
print(f"  - Historial:         {len(historial_file_names)} archivo(s)")
if other_file_names:
    print(f"  - Otros:             {other_file_names}")
print(f"\nTotal: {len(raw_file_paths)} archivos")

Archivos en data/raw:
  - alarmReport_*.xlsx: 2 archivos
  - dls_report_*.xlsx:  56 archivos (fuente de Procesamiento de Datos)
  - Historial:         1 archivo(s)

Total: 59 archivos


## 2. Estructura de los archivos `dls_report_*.xlsx`

Los archivos **dls_report** contienen las sesiones de registro de datos con lecturas de gases. La tabla de datos tiene cabecera en la **fila 24** (0-based) con columnas: Descripción (código gas en col 2), Sensor (col 0), Número de período (col 1), Gas, Leyendo, Unidad, STEL Máx, TWA máximo, Tiempo de inicio, Tiempo de finalización, Sitio, Factor de respuesta, Factor, Alarma, Temperatura.

In [2]:
# Inspección de un archivo dls_report (muestra)
sample_dls_report_path = next(RAW_DIR.glob("dls_report_*.xlsx"), None)
if sample_dls_report_path is not None:
    xl = pd.ExcelFile(sample_dls_report_path)
    print(f"Archivo: {sample_dls_report_path.name}")
    print(f"Hojas: {xl.sheet_names}")
    df_sample = pd.read_excel(
        sample_dls_report_path,
        sheet_name=DLS_SHEET_NAME,
        header=DLS_HEADER_ROW,
    )
    print(f"\nColumnas (header fila {DLS_HEADER_ROW}): {list(df_sample.columns)}")
    print(f"Filas de datos: {len(df_sample)}")
    reading_numeric = pd.to_numeric(df_sample[COL_READING], errors="coerce")
    rows_with_numeric_reading = df_sample.loc[reading_numeric.notna()]
    print(f"Filas con {COL_READING} numérica: {len(rows_with_numeric_reading)}")
    if len(rows_with_numeric_reading) > 0:
        print("\nMuestra (primeras filas con datos):")
        display(rows_with_numeric_reading.head())

Archivo: dls_report_4101600256597.xlsx
Hojas: ['Resumen']

Columnas (header fila 24): ['Descripción', 'Unnamed: 1', 'Unnamed: 2', 'Gas', 'Leyendo', 'Unidad', 'STEL Máx', 'TWA máximo', 'Tiempo de inicio', 'Tiempo de finalización', 'Sitio', 'Factor de respuesta', 'Factor', 'Alarma', 'Temperatura']
Filas de datos: 56
Filas con Leyendo numérica: 40

Muestra (primeras filas con datos):


,Descripción,Unnamed: 1,Unnamed: 2,Gas,Leyendo,Unidad,STEL Máx,TWA máximo,Tiempo de inicio,Tiempo de finalización,Sitio,Factor de respuesta,Factor,Alarma,Temperatura
2,NaN,NaN,NO2,Dióxido de nitrógeno,-0.6,ppm,NaN,NaN,2023-03-31 14:37:39,2023-03-31 14:38:29,NaN,NaN,NaN,NaN,20.0
3,NaN,NaN,NO2,Dióxido de nitrógeno,-0.3,ppm,NaN,NaN,2023-03-31 14:37:29,2023-03-31 14:37:39,NaN,NaN,NaN,NaN,20.0
4,NaN,NaN,NO2,Dióxido de nitrógeno,-0.1,ppm,NaN,NaN,2023-03-31 14:37:19,2023-03-31 14:37:29,NaN,NaN,NaN,NaN,20.0
5,NaN,NaN,NO2,Dióxido de nitrógeno,0.0,ppm,NaN,NaN,2023-03-31 14:37:09,2023-03-31 14:37:19,NaN,NaN,NaN,NaN,20.0
7,NaN,NaN,NO2,Dióxido de nitrógeno,-0.5,ppm,NaN,NaN,2023-03-31 14:40:19,2023-03-31 14:41:29,NaN,NaN,NaN,NaN,20.0


## 3. Función para leer y normalizar un archivo `dls_report`

Se lee la hoja "Resumen" con cabecera en fila 24, se rellenan Sensor y Número de período hacia abajo, y se filtran solo las filas con `Leyendo` numérica. La columna de código de gas (O2, NO2, CO, etc.) está en la tercera columna del Excel.

In [3]:
def cargar_dls_report(ruta: Path) -> pd.DataFrame:
    """
    Carga un archivo dls_report_*.xlsx y devuelve un DataFrame normalizado
    con el mismo esquema que Procesamiento de Datos (compatible con el ETL).
    """
    df = pd.read_excel(ruta, sheet_name=DLS_SHEET_NAME, header=DLS_HEADER_ROW)
    # Nombre de la columna "Descripción" puede venir con encoding distinto
    col_desc = [c for c in df.columns if "Descripci" in str(c) or c == "Descripción"][:1]
    col_unnamed1 = [c for c in df.columns if "Unnamed: 1" in str(c)][:1]
    col_unnamed2 = [c for c in df.columns if "Unnamed: 2" in str(c)][:1]

    # Rellenar Sensor: filas donde la primera columna contiene "SN:"
    sensor_col = df.columns[0]
    mask_sensor = df[sensor_col].astype(str).str.contains("SN:", na=False)
    df["Sensor"] = df[sensor_col].where(mask_sensor).ffill()
    # Número de período
    if col_unnamed1:
        mask_per = df[col_unnamed1[0]].astype(str).str.contains("per", na=False)
        df["Número de Periodo"] = df[col_unnamed1[0]].where(mask_per).ffill()
    else:
        df["Número de Periodo"] = None
    # Descripción = código del gas (O2, NO2, CO, etc.) en columna 3
    if col_unnamed2:
        df["Descripción"] = df[col_unnamed2[0]]
    elif col_desc:
        df["Descripción"] = df[col_desc[0]]

    # Columna de lectura numérica (decimal con coma → float; inválidos → NaN)
    df[COL_READING] = pd.to_numeric(
        df[COL_READING].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )
    df = df.dropna(subset=[COL_READING])
    if df.empty:
        return df

    # Columnas finales con nombres esperados por el ETL
    renombres = {
        "Tiempo de inicio": "Tiempo de Inicio",
        "Tiempo de finalización": "Tiempo de finalización",
        "Tiempo de finalizaci\u00f3n": "Tiempo de finalización",
        "Factor de respuesta": "Factor de Respuesta",
        "STEL M\u00e1x": "STEL Máx",
        "TWA m\u00e1ximo": "TWA máximo",
    }
    df = df.rename(columns={k: v for k, v in renombres.items() if k in df.columns})

    columnas_etl = [
        "Descripción", "Sensor", "Número de Periodo", "Gas", COL_READING, "Unidad",
        "STEL Máx", "TWA máximo", "Tiempo de Inicio", "Tiempo de finalización",
        "Sitio", "Factor de Respuesta", "Factor", "Alarma", "Temperatura",
    ]
    out = df[[c for c in columnas_etl if c in df.columns]].copy()
    out["Tiempo de Inicio"] = pd.to_datetime(out["Tiempo de Inicio"], errors="coerce")
    out["Tiempo de finalización"] = pd.to_datetime(out["Tiempo de finalización"], errors="coerce")
    return out

In [4]:
# Normalizar "Número de Periodo" a formato "Periodo N" (opcional, para consistencia con ETL)
def _normalizar_periodo(val):
    if pd.isna(val):
        return val
    s = str(val).strip()
    if "per" in s.lower() and ":" in s:
        num = s.split(":")[-1].strip()
        return f"Periodo {num}" if num.isdigit() else s
    return s

## 4. Cargar todos los `dls_report`, concatenar y generar CSV por año-mes

Se procesan solo archivos `dls_report_*.xlsx` (se excluyen duplicados con sufijo " (1)", " (2)" si se desea). Se agrupa por año-mes según `Tiempo de Inicio` y se guarda un CSV por mes en `data/processed`.

### Verificación: no sobrescribir archivos ya existentes en `data/processed`

Solo se escriben los CSV que **no existan** en `data/processed`. Si el archivo ya existe, se omite (no se crea ni se sobrescribe).

In [5]:
# Listar dls_report (opcional: excluir duplicados " (1)", " (2)")
dls_paths = sorted(RAW_DIR.glob("dls_report_*.xlsx"))
# Excluir variantes duplicadas que tienen " (1)" o " (2)" en el nombre
dls_paths = [p for p in dls_paths if " (" not in p.name]
print(f"Archivos dls_report a procesar: {len(dls_paths)}")

Archivos dls_report a procesar: 50


In [6]:
listas_dfs = []
errores = []
for ruta in dls_paths:
    try:
        df_one = cargar_dls_report(ruta)
        if not df_one.empty:
            listas_dfs.append(df_one)
    except Exception as e:
        errores.append((ruta.name, str(e)))
if errores:
    print("Errores al cargar:", errores)
df_full = pd.concat(listas_dfs, ignore_index=True) if listas_dfs else pd.DataFrame()
if df_full.empty:
    print("No se cargó ningún dato.")
else:
    df_full["Número de Periodo"] = df_full["Número de Periodo"].apply(_normalizar_periodo)
    df_full = df_full.sort_values("Tiempo de Inicio").reset_index(drop=True)
    print(f"Total filas concatenadas: {len(df_full)}")
    print(f"Rango de fechas: {df_full['Tiempo de Inicio'].min()} a {df_full['Tiempo de Inicio'].max()}")

/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default sty

/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


... (4 outputs left out)

/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/usr/local/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Total filas concatenadas: 13894
Rango de fechas: 2023-03-31 14:37:09 a 2025-12-11 14:17:44


### Verificación: año-meses en los datos (no solo 2023-03 y 2024-09)

Se comprueba si hay datos de años anteriores, posteriores y de más meses.

In [7]:
# Verificación de rango temporal en los datos cargados desde data/raw
if not df_full.empty:
    df_full["año_mes"] = df_full["Tiempo de Inicio"].dt.to_period("M")
    meses_unicos = sorted(df_full["año_mes"].unique())
    años_unicos = sorted(df_full["Tiempo de Inicio"].dt.year.unique())
    fecha_min = df_full["Tiempo de Inicio"].min()
    fecha_max = df_full["Tiempo de Inicio"].max()

    print("Resumen temporal (datos en data/raw):")
    print(f"  Fecha mínima: {fecha_min}")
    print(f"  Fecha máxima: {fecha_max}")
    print(f"  Año-meses presentes: {[str(m) for m in meses_unicos]}")
    print(f"  Total de meses con datos: {len(meses_unicos)}")
    print()

    hay_antes_2023 = any(y < 2023 for y in años_unicos)
    hay_despues_2024 = any(y > 2024 for y in años_unicos)
    solo_2023_03_y_2024_09 = set(meses_unicos) <= {pd.Period("2023-03"), pd.Period("2024-09")}

    if hay_antes_2023:
        print("  Hay datos de años ANTERIORES a 2023.")
    else:
        print("  No hay datos de años anteriores a 2023.")
    if hay_despues_2024:
        print("  Hay datos de años POSTERIORES a 2024.")
    else:
        print("  No hay datos de años posteriores a 2024.")
    if solo_2023_03_y_2024_09:
        print("  Los datos están SOLO en 2023-03 y 2024-09.")
    else:
        print("  Los datos NO están solo en 2023-03 y 2024-09; hay más meses y/o años.")
else:
    print("No hay datos cargados para verificar.")

Resumen temporal (datos en data/raw):
  Fecha mínima: 2023-03-31 14:37:09
  Fecha máxima: 2025-12-11 14:17:44
  Año-meses presentes: ['2023-03', '2024-09', '2024-11', '2024-12', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12']
  Total de meses con datos: 9

  No hay datos de años anteriores a 2023.
  Hay datos de años POSTERIORES a 2024.
  Los datos NO están solo en 2023-03 y 2024-09; hay más meses y/o años.


In [8]:
# Crear carpeta data/processed si no existe
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Agrupar por año-mes y guardar un CSV por mes (solo si el archivo NO existe)
if not df_full.empty:
    df_full["año_mes"] = df_full["Tiempo de Inicio"].dt.to_period("M")
    archivos_guardados = []
    archivos_omitidos = []
    for periodo, grupo in df_full.groupby("año_mes", sort=True):
        ym = str(periodo)
        nombre = f"Procesamiento_de_Datos_{ym}.csv"
        ruta_out = PROCESSED_DIR / nombre
        if ruta_out.exists() and not OVERWRITE_PROCESSED_FILES:
            archivos_omitidos.append((nombre, len(grupo)))
            continue
        grupo_out = grupo.drop(columns=["año_mes"])
        grupo_out.to_csv(ruta_out, index=False, encoding="utf-8-sig")
        archivos_guardados.append((nombre, len(grupo)))
    print("Archivos guardados en data/processed (nuevos):")
    for nombre, filas in archivos_guardados:
        print(f"  - {nombre} ({filas} filas)")
    if archivos_omitidos:
        print("\nArchivos omitidos (ya existían, no se sobrescriben):")
        for nombre, filas in archivos_omitidos:
            print(f"  - {nombre} ({filas} filas)")
else:
    print("No hay datos para guardar.")

Archivos guardados en data/processed (nuevos):

Archivos omitidos (ya existían, no se sobrescriben):
  - Procesamiento_de_Datos_2023-03.csv (40 filas)
  - Procesamiento_de_Datos_2024-09.csv (513 filas)
  - Procesamiento_de_Datos_2024-11.csv (8937 filas)
  - Procesamiento_de_Datos_2024-12.csv (18 filas)
  - Procesamiento_de_Datos_2025-08.csv (236 filas)
  - Procesamiento_de_Datos_2025-09.csv (322 filas)
  - Procesamiento_de_Datos_2025-10.csv (184 filas)
  - Procesamiento_de_Datos_2025-11.csv (2344 filas)
  - Procesamiento_de_Datos_2025-12.csv (1300 filas)


## 5. (Opcional) Archivo consolidado

Un único CSV con todos los datos permite compatibilidad con el notebook ETL que lee `Procesamiento de Datos.xlsx`. Puedes usar `Procesamiento_de_Datos.csv` en lugar del Excel.

In [9]:
# Guardar consolidado (respeta OVERWRITE_PROCESSED_FILES en .env)
if not df_full.empty:
    ruta_consolidado = PROCESSED_DIR / "Procesamiento_de_Datos.csv"
    if not ruta_consolidado.exists() or OVERWRITE_PROCESSED_FILES:
        consolidado = df_full.drop(columns=["año_mes"])
        consolidado.to_csv(ruta_consolidado, index=False, encoding="utf-8-sig")
        print(f"Archivo consolidado guardado: {ruta_consolidado} ({len(consolidado)} filas)")
    else:
        print(f"Consolidado ya existe, no se sobrescribe: {ruta_consolidado}")

Consolidado ya existe, no se sobrescribe: /app/data/processed/Procesamiento_de_Datos.csv
